# Notebook7 - Root Cause Analysis with LLM

整合 reduced alerts、metric interaction rules 與事件視窗 evidence，產生 RCA candidate ranking 和 LLM prompt。

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

def show_fig(fig):
    display(fig)
    plt.close(fig)

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = Path("..").resolve()

DATA_SYNTHETIC = PROJECT_ROOT / "data" / "synthetic"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
REPORTS = PROJECT_ROOT / "reports"
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
REPORTS.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
features = pd.read_csv(DATA_PROCESSED / "features.csv", parse_dates=["timestamp"])
reduced = pd.read_csv(DATA_PROCESSED / "reduced_alerts.csv", parse_dates=["start_time", "end_time"])
print(reduced.shape)
display(reduced.head())

## Step 1 - 建立 RCA rules

把 reduced alert 的 evidence metrics 轉成 root cause candidate。

In [ ]:
def candidate_from_row(row):
    metrics = set(str(row.get("evidence_metrics", "")).split(","))
    problem = row.get("problem_type", "General anomaly")
    if "Queue congestion" in problem or "Queue / buffer" in problem:
        return "Queue congestion / buffer pressure", [
            "OCTETS high", "DISCARDS high", "ERRORS normal or not dominant",
            "Check QoS, queue, bandwidth, top talkers, scheduled backup jobs",
        ]
    if "Broadcast" in problem:
        return "Broadcast storm / L2 loop", [
            "BROADCAST high across ports", "NUCAST high may rise together",
            "Check STP, loop, ARP storm, DHCP storm, VLAN scope",
        ]
    if "Multicast" in problem:
        return "Multicast flooding", [
            "MULTICAST high across ports", "DISCARDS may rise",
            "Check IGMP snooping, querier, IPTV or routing protocol behavior",
        ]
    if "Link quality" in problem or "errors" in metrics:
        return "Link quality issue", [
            "ERRORS spike or repeated spikes", "Traffic may not be high",
            "Check cable, SFP, NIC, port, duplex mismatch",
        ]
    if "unknown_protocol" in metrics:
        return "Unknown protocol / scan", [
            "INUNKNOWNPROTOS high", "INPKTS high",
            "Check non-standard protocol, scan, attack, VLAN/routing config, new device rollout",
        ]
    if "Packet surge" in problem:
        return "Small packet scan or connection surge", [
            "UCASTPKTS high", "OCTETS may be flat or only mildly high",
            "Check port scan, short connections, DDoS small packets, heartbeat surge",
        ]
    if "Traffic surge" in problem:
        return "Business traffic growth or large transfer", [
            "OCTETS high", "Compare packet growth and avg_packet_size",
            "Check backup, data sync, download, upload, streaming, or capacity pressure",
        ]
    return "General network anomaly", ["Review metric trend, affected ports, recent changes"]

## Step 2 - 產生 RCA event table 與 scores

In [ ]:
rca_rows = []
for idx, row in reduced.iterrows():
    candidate, evidence = candidate_from_row(row)
    duration_minutes = max(5, int((row["end_time"] - row["start_time"]).total_seconds() / 60) + 5)
    score = min(100, int(row["severity_score"]) + 3 * len(evidence) + row["affected_port_count"] * 4)
    rca_rows.append(
        {
            "rca_event_id": f"RCA-{idx + 1:03d}",
            "start_time": row["start_time"],
            "end_time": row["end_time"],
            "affected_ports": row["affected_ports"],
            "problem_type": row["problem_type"],
            "root_cause_candidate": candidate,
            "evidence": " | ".join(evidence),
            "duration_minutes": duration_minutes,
            "severity_score": row["severity_score"],
            "root_cause_score": score,
            "confidence": "High" if score >= 85 else "Medium" if score >= 65 else "Low",
        }
    )

rca = pd.DataFrame(rca_rows).sort_values(["root_cause_score", "start_time"], ascending=[False, True])
display(rca.head(12))

## Step 3 - 視覺化：Root cause candidate ranking 與 confidence

In [ ]:
candidate_summary = (
    rca.groupby("root_cause_candidate")
    .agg(events=("rca_event_id", "count"), avg_score=("root_cause_score", "mean"), max_score=("root_cause_score", "max"))
    .sort_values("events", ascending=False)
)
display(candidate_summary)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
candidate_summary["events"].sort_values().plot(kind="barh", ax=axes[0], color="tab:blue")
axes[0].set_title("RCA candidate event count")
axes[0].set_xlabel("events")

rca["confidence"].value_counts().reindex(["Low", "Medium", "High"]).fillna(0).plot(kind="bar", ax=axes[1], color="tab:green")
axes[1].set_title("Confidence distribution")
axes[1].set_ylabel("events")
plt.tight_layout()
show_fig(fig)

## Step 4 - 產生 LLM prompt 與 RCA report

In [ ]:
def build_llm_prompt(event):
    lines = [
        "你是一位 AIOps / Network RCA 助理。",
        "請根據以下監控證據判斷可能根因，並輸出：",
        "1. 問題型態",
        "2. 主要證據",
        "3. 可能根因",
        "4. 建議處置",
        "5. 信心分數",
        "",
        f"事件時間：{event.start_time} - {event.end_time}",
        f"影響介面：{event.affected_ports}",
        f"問題類型：{event.problem_type}",
        f"RCA 候選：{event.root_cause_candidate}",
        f"證據：{event.evidence}",
        f"嚴重度：{event.severity_score}",
        f"信心：{event.confidence}",
    ]
    return "\n".join(lines)


report_lines = [
    "# RCA Report",
    "",
    "本報告由 Notebook7 根據 reduced alerts 與 metric interaction rules 產生，可作為 LLM prompt 或人工診斷摘要。",
    "",
]
for event in rca.head(12).itertuples(index=False):
    report_lines.extend(
        [
            f"## {event.rca_event_id} - {event.root_cause_candidate}",
            "",
            f"- 事件時間：{event.start_time} - {event.end_time}",
            f"- 影響介面：{event.affected_ports}",
            f"- 問題型態：{event.problem_type}",
            f"- 主要證據：{event.evidence}",
            f"- 根因分數：{event.root_cause_score}",
            f"- 信心分數：{event.confidence}",
            "",
            "### LLM Prompt",
            "",
            "```text",
            build_llm_prompt(event).strip(),
            "```",
            "",
        ]
    )

out_events = DATA_PROCESSED / "rca_events.csv"
out_report = REPORTS / "rca_report.md"
rca.to_csv(out_events, index=False)
out_report.write_text("\n".join(report_lines), encoding="utf-8")
print(f"saved: {out_events}")
print(f"saved: {out_report}")
print("\n".join(report_lines[:30]))

## Step 5 - RCA workflow 回顧

Metrics → Time Window → Anomaly Flags → Metric Interaction Rules → Root Cause Candidate Ranking → LLM Explanation → Recommended Action